In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import json
import sys
from datetime import date
from pathlib import Path

import mne
import numpy as np
import optuna
import torch
import yaml
from torch.utils.data import DataLoader

root = Path.cwd().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from lib.data import TFRDataset
from lib.models.alexnet import AlexNetTFR
from lib.models.tfr_transformer import TFRTransformerWrapper
from lib.optuna import (
    attrs_fn,
    cumulative_loss_metric_factory,
    loss_slope,
    make_objective_engine,
    make_splits_fn_factory,
    objectives_fn,
    params_fn_factory,
    params_fn_factory_transformer,
    run_fold_fn_factory,
)
from lib.training.epochs import eval_one_epoch_f1_macro, train_one_epoch
from lib.data.normalisation import normalize_tfr_robust

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("project root:", root)
print("device:", device)

In [ ]:
import warnings
warnings.filterwarnings('ignore') # Suppress all warnings

In [ ]:
# Рабочая директория: .../Pirogov/NeuronDeCo/notebooks
NB_DIR = Path.cwd().resolve()
PROJECT_ROOT = NB_DIR.parent
PIROGOV_ROOT = PROJECT_ROOT.parent
PREPROCESSED_ROOT = PIROGOV_ROOT / "PreprocessedData"

patients_candidates = [
    PREPROCESSED_ROOT / "patients.yaml",  # приоритет: как вы просили
    PROJECT_ROOT / "patients.yaml",       # fallback на старый вариант
]
patients_yaml = next((p for p in patients_candidates if p.exists()), None)
if patients_yaml is None:
    raise FileNotFoundError(
        "patients.yaml not found. Checked: " + ", ".join(str(p) for p in patients_candidates)
    )

with open(patients_yaml, encoding="utf-8") as f:
    test = yaml.safe_load(f)
pat_config = test["default"]

min_f, max_f, min_t, max_t = (
    pat_config["min_f"],
    pat_config["max_f"],
    pat_config["min_t"],
    pat_config["max_t"],
)

seed = 42
cv = True
test_size = 0.2
cv_aggregate = "median"
max_epochs = 100
patience = 10
n_trials = 100

cumulative_weighted_loss_metric = cumulative_loss_metric_factory(
    up_weight=1.1,
    down_weight=1.0,
)

for s in ['s02','s03','s04','s05','s06','s07','s09','s10','s11','s12','s13','s15']: # Хардкод выбора пациентов 
    # time_frequency_file = Path(pat_config["time_frequency_file"])
    tfr_candidates = [
        PREPROCESSED_ROOT / "specs_with_car" / f"tfr_{s}.fif",  # приоритет: PreprocessedData
        PROJECT_ROOT / "specs_with_car" / f"tfr_{s}.fif",       # fallback
    ]
    time_frequency_file = next((p for p in tfr_candidates if p.exists()), None)
    if time_frequency_file is None:
        raise FileNotFoundError(
            "TFR file not found. Checked: " + ", ".join(str(p) for p in tfr_candidates)
        )
    DATA_ROOT_ROOT = time_frequency_file.parent.parent #  Хардкод структуры
    DATE_TAG = date.today().isoformat()
    OPTUNA_DB_DIR = DATA_ROOT_ROOT / DATE_TAG
    OPTUNA_DB_DIR.mkdir(parents=True, exist_ok=True)

    print("notebooks dir:", NB_DIR)
    print("Pirogov root:", PIROGOV_ROOT)
    print("PreprocessedData root:", PREPROCESSED_ROOT)
    print("patients.yaml:", patients_yaml)
    print("TFR file:", time_frequency_file)
    print("Optuna DB dir:", OPTUNA_DB_DIR)
    print("config slices:", (min_f, max_f, min_t, max_t))

    tfr = mne.time_frequency.read_tfrs(str(time_frequency_file))[0]
    y = np.where(tfr.events[:, 2] == 9, 1, 0).astype(np.int64)

    # X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, min_f:max_f, min_t:max_t]
    X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, :-50, :].astype(np.float32)

    del tfr
    gc.collect()  # Теперь память гарантировано свободна

    n, c, f, t = X.shape
    num_classes = int(np.unique(y).shape[0])
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("num_classes:", num_classes)

    # ==================================
    # Alexnet
    # ==================================

    alex_db = OPTUNA_DB_DIR / f"{time_frequency_file.stem}_alexnet.db"
    alex_storage = f"sqlite:///{alex_db}"

    objective_alex = make_objective_engine(
        X=X,
        y=y,
        make_splits_fn=make_splits_fn_factory(test_size=test_size, seed=seed, cv=cv),
        run_fold_fn=run_fold_fn_factory(
            ModelCls=AlexNetTFR,
            device=device,
            max_epochs=max_epochs,
            patience=patience,
            TFRDataset=TFRDataset,
            DataLoader=DataLoader,
            train_one_epoch=train_one_epoch,
            eval_one_epoch_f1_macro=eval_one_epoch_f1_macro,
            loss_metric=cumulative_weighted_loss_metric,
        ),
        aggregate_mode=cv_aggregate,
        params_fn=params_fn_factory(in_channels=c, num_classes=num_classes),
        objectives_fn=objectives_fn,
        attrs_fn=attrs_fn,
    )

    study_a = optuna.create_study(
        directions=["maximize", "minimize"],
        sampler=optuna.samplers.NSGAIISampler(seed=seed),
        storage=alex_storage,
        study_name=f"{time_frequency_file.stem}_alexnet",
        load_if_exists=True,
    )
    study_a.optimize(objective_alex, n_trials=n_trials, show_progress_bar=True)

    print("AlexNet DB:", alex_db)

    # ==================================
    # Transformer
    # ==================================
    tr_db = OPTUNA_DB_DIR / f"{time_frequency_file.stem}_transformer.db"
    tr_storage = f"sqlite:///{tr_db}"

    objective_tr = make_objective_engine(
        X=X,
        y=y,
        make_splits_fn=make_splits_fn_factory(test_size=test_size, seed=seed, cv=cv),
        run_fold_fn=run_fold_fn_factory(
            ModelCls=TFRTransformerWrapper,
            device=device,
            max_epochs=max_epochs,
            patience=patience,
            TFRDataset=TFRDataset,
            DataLoader=DataLoader,
            train_one_epoch=train_one_epoch,
            eval_one_epoch_f1_macro=eval_one_epoch_f1_macro,
            loss_metric=cumulative_weighted_loss_metric,
        ),
        aggregate_mode=cv_aggregate,
        params_fn=params_fn_factory_transformer(
            num_classes=num_classes,
            seq_len=t,
        ),
        objectives_fn=objectives_fn,
        attrs_fn=attrs_fn,
    )

    study_t = optuna.create_study(
        directions=["maximize", "minimize"],
        sampler=optuna.samplers.NSGAIISampler(seed=seed + 1),
        storage=tr_storage,
        study_name=f"{time_frequency_file.stem}_transformer",
        load_if_exists=True,
    )
    study_t.optimize(objective_tr, n_trials=n_trials, show_progress_bar=True)

    print("Transformer DB:", tr_db)